# ARC-AGI-2 E4：提高推理上限（96k）的自然結束率量測（不是提交）

不跑 NVARC。以 v2 的設定（Qwen3.8-27B-FP8、vLLM 0.19.0、TP4、MTP2、無預算強制），但 cap 96k、併發依 KV 公式，在公開 eval 的 keys30 中 E3／v2 從未自然結束的題目上跑 10 個新種子（rep 3）工作；每列以 eval 解答評分。量的是 64k 之後到 80k／96k 之間自然結束的比例（預登記 prereg-e4-cap96k-v1.md）。

In [ ]:
import time
global_start = time.time()

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%writefile arc_llm.py
"""Pure helpers for the Qwen3.8 ARC probe (new-approach-proposal-v1 §6): prompt, answer parsing, dihedral + colour
augmentation and its inverse, and summary statistics. No I/O, no third-party imports, so it runs in the probe and in
local unit tests alike."""
import json
import math
import random
import re
import zlib

# --- grids -----------------------------------------------------------------------------------------------------------

def valid_grid(g):
    return (isinstance(g, list) and 0 < len(g) <= 30 and all(isinstance(r, list) for r in g)
            and 0 < len(g[0]) <= 30 and all(len(r) == len(g[0]) for r in g)
            and all(type(v) is int and 0 <= v <= 9 for r in g for v in r))


def transpose(g):
    return [list(r) for r in zip(*g)]


def rot90(g):  # clockwise
    return [list(r) for r in zip(*g[::-1])]


def dihedral(g, k):
    """k in 0..7: rotate k%4 quarter turns clockwise, then transpose if k >= 4."""
    for _ in range(k % 4):
        g = rot90(g)
    return transpose(g) if k >= 4 else [list(r) for r in g]


def dihedral_inverse(g, k):
    if k >= 4:
        g = transpose(g)
    for _ in range((4 - k % 4) % 4):
        g = rot90(g)
    return [list(r) for r in g]


def recolor(g, perm):
    return [[perm[v] for v in r] for r in g]


def augmentation(task_id, salt="p0-v1"):
    """Deterministic non-identity transform per task: dihedral k in 1..7 and a permutation of colours 1..9 (0 fixed,
    ARC background)."""
    rng = random.Random(zlib.crc32(f"{salt}:{task_id}".encode()))
    k = rng.randrange(1, 8)
    rest = list(range(1, 10))
    rng.shuffle(rest)
    perm = [0] + rest
    return {"k": k, "perm": perm}


def apply_aug(grid, aug):
    return recolor(dihedral(grid, aug["k"]), aug["perm"])


def invert_aug(grid, aug):
    inv = [0] * 10
    for a, b in enumerate(aug["perm"]):
        inv[b] = a
    return dihedral_inverse(recolor(grid, inv), aug["k"])


def augment_task(task, aug):
    return {"train": [{"input": apply_aug(p["input"], aug), "output": apply_aug(p["output"], aug)}
                      for p in task["train"]],
            "test": [{"input": apply_aug(p["input"], aug)} for p in task["test"]]}


# --- prompt / parse --------------------------------------------------------------------------------------------------

def grid_text(g):
    return "[" + ",\n ".join(json.dumps(r, separators=(",", ":")) for r in g) + "]"


def build_prompt(task, test_index):
    parts = ["You are solving an ARC-AGI puzzle. Each example maps an input grid to an output grid with the same "
             "hidden rule. Grids are lists of rows; each cell is a colour 0-9.", ""]
    for i, p in enumerate(task["train"], 1):
        parts += [f"Example {i} input:", grid_text(p["input"]), f"Example {i} output:", grid_text(p["output"]), ""]
    parts += ["Test input:", grid_text(task["test"][test_index]["input"]), "",
              "Find the rule, apply it to the test input, and give the test output grid. End your answer with the "
              "output grid as a JSON list of lists inside a ```json code block, and nothing after it."]
    return "\n".join(parts)


_FENCE = re.compile(r"```(?:json)?\s*(\[.*?\])\s*```", re.S)


def _candidates(text):
    for m in _FENCE.finditer(text):
        yield m.group(1)
    # bare outermost [[...]] blocks, last ones last
    depth, start = 0, None
    for i, ch in enumerate(text):
        if ch == "[":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "]" and depth:
            depth -= 1
            if depth == 0 and start is not None:
                yield text[start:i + 1]


def parse_answer(text):
    """Last valid grid found, scanning fenced blocks first and then every bare outermost list, so a valid bare list after
    the fence wins; None if there is none."""
    if not text:
        return None
    found = None
    for chunk in _candidates(text):
        try:
            g = json.loads(chunk)
        except (ValueError, RecursionError):
            continue
        if valid_grid(g):
            found = g
    return found


# --- statistics ------------------------------------------------------------------------------------------------------

def quantiles(xs, qs=(0.5, 0.9)):
    if not xs:
        return {f"p{int(q * 100)}": None for q in qs}
    s = sorted(xs)
    return {f"p{int(q * 100)}": s[min(len(s) - 1, max(0, math.ceil(q * len(s)) - 1))] for q in qs}


def km_quantiles(done, cens, qs=(0.5, 0.9)):
    """Kaplan-Meier quantiles of tokens-per-attempt with right-censoring (aborted/unfinished attempts are censored at
    the tokens they reached). None where the survival curve never drops that far."""
    events = sorted([(t, 1) for t in done] + [(t, 0) for t in cens])
    at_risk, surv, out, i = len(events), 1.0, {}, 0
    pending = list(qs)
    while i < len(events) and pending:
        t = events[i][0]
        d = c = 0
        while i < len(events) and events[i][0] == t:
            d += events[i][1]
            c += 1 - events[i][1]
            i += 1
        if d and at_risk:
            surv *= 1 - d / at_risk
        at_risk -= d + c
        while pending and 1 - surv >= pending[0]:
            out[f"km_p{int(pending.pop(0) * 100)}"] = t
    for q in pending:
        out[f"km_p{int(q * 100)}"] = None
    return out


def token_stats(rows):
    done = [r["completion_tokens"] for r in rows if r.get("state") == "done" and r.get("completion_tokens")]
    cens = [r["completion_tokens"] for r in rows if r.get("state") != "done" and r.get("completion_tokens")]
    out = {"n": len(rows), "n_done": len(done), "n_censored": len(cens),
           "mean_done": (sum(done) / len(done)) if done else None, "max_done": max(done) if done else None,
           "sum_all": sum(done) + sum(cens), "censored_over_40k": sum(1 for t in cens if t >= 40000),
           "censored_over_80k": sum(1 for t in cens if t >= 80000)}
    out.update(quantiles(done))
    out.update(km_quantiles(done, cens))
    return out


def arm_summary(rows):
    """rows: one effort arm. Accuracy is per test output; truncation = finish_reason 'length' among finished."""
    finished = [r for r in rows if r.get("state") == "done"]
    return {"tokens": token_stats(rows),
            "truncated": sum(1 for r in finished if r.get("finish_reason") == "length"),
            "parsed": sum(1 for r in finished if r.get("answer") is not None),
            "correct": sum(1 for r in finished if r.get("correct")),
            "finished": len(finished), "requests": len(rows)}


def pair_agreement(rows):
    """For an arm sampled twice per output: agreement count and its precision (agreeing answer correct)."""
    by = {}
    for r in rows:
        if r.get("state") == "done":
            by.setdefault(r["key"], []).append(r)
    agree = agree_ok = both = 0
    for rs in by.values():
        if len(rs) < 2:
            continue
        both += 1
        a, b = rs[0].get("answer"), rs[1].get("answer")
        if a is not None and a == b:
            agree += 1
            agree_ok += bool(rs[0].get("correct"))
    return {"pairs": both, "agree": agree, "agree_correct": agree_ok,
            "precision": (agree_ok / agree) if agree else None}


In [ ]:
%%writefile progsynth.py
"""Program-synthesis channel helpers (CSR-arc-agi-2-001 #1/#3, HANDOFF 10-13; v13: sandbox hardened per API review 20261013T150520Z): prompt for a `transform(grid)` program,
extract it, run it in a resource-limited subprocess on the train inputs and the test inputs, and build one round of
execution feedback. Pure stdlib; the sandbox is a separate `python -I` process per program."""
import json
import re
import subprocess
import sys
import textwrap

import arc_llm as L

RUNNER = r'''
import json, os, resource, sys
for lim, val in ((resource.RLIMIT_AS, 2 << 30), (resource.RLIMIT_CPU, 20), (resource.RLIMIT_FSIZE, 10 << 20)):
    try:
        resource.setrlimit(lim, (val, val))
    except (ValueError, OSError):  # macOS refuses RLIMIT_AS; Linux (Colab/Kaggle) applies it
        pass
real = os.fdopen(os.dup(1), "w")  # the result channel; the model's own prints go to /dev/null (no parent OOM)
null = open(os.devnull, "w")
os.dup2(null.fileno(), 1)
os.dup2(null.fileno(), 2)
sys.stdout = sys.stderr = null
src, inputs = json.loads(sys.stdin.read())
ns = {}
out = []
try:
    exec(compile(src, "<model>", "exec"), ns)
    fn = ns["transform"]
except BaseException as e:
    real.write(json.dumps({"compile_error": repr(e)[:400]})); real.flush(); os._exit(0)
for g in inputs:
    try:
        r = fn([list(row) for row in g])
        if hasattr(r, "tolist"):
            r = r.tolist()
        r = [[int(v) for v in row] for row in r]
        out.append({"ok": r})
    except BaseException as e:
        out.append({"err": repr(e)[:300]})
real.write(json.dumps({"results": out})); real.flush()
os._exit(0)
'''


def build_prompt(task):
    parts = ["You are solving an ARC-AGI puzzle by writing a program. Each example maps an input grid to an output grid "
             "with the same hidden rule. Grids are lists of rows; each cell is a colour 0-9.", ""]
    for i, p in enumerate(task["train"], 1):
        parts += [f"Example {i} input:", L.grid_text(p["input"]), f"Example {i} output:", L.grid_text(p["output"]), ""]
    for i, p in enumerate(task["test"], 1):
        parts += [f"Test input {i} (output unknown):", L.grid_text(p["input"]), ""]
    parts += ["Write a Python function `transform(grid)` that takes one input grid (list of lists of int) and returns "
              "the output grid (list of lists of int). It must reproduce every example output exactly and generalise "
              "to the test inputs. You may use the standard library and numpy. Give the complete final code in one "
              "```python code block at the end of your answer."]
    return "\n".join(parts)


_CODE = re.compile(r"```(?:python|py)?\s*\n(.*?)```", re.S)


def extract_code(text):
    blocks = [b for b in _CODE.findall(text or "") if "def transform" in b]
    return blocks[-1] if blocks else None


def run_program(code, grids, timeout=30):
    """Returns (results, error): results is a list of {"ok": grid} / {"err": msg} per input grid. The child runs in its
    own session and an empty temp cwd, single-threaded BLAS, and is killed with its whole process group on timeout."""
    import os
    import shutil
    import signal
    import tempfile
    cwd = tempfile.mkdtemp(prefix="v13prog-")
    env = dict(os.environ, OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
    p = None
    try:
        # -s, not -I: keep the notebook's PYTHONPATH (Kaggle's numpy may live there); the cwd is an empty temp dir, so
        # nothing can be planted next to the runner
        p = subprocess.Popen([sys.executable, "-s", "-c", RUNNER], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                             stderr=subprocess.DEVNULL, text=True, cwd=cwd, env=env, start_new_session=True)
        stdout, _ = p.communicate(json.dumps([code, grids]), timeout=timeout)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(p.pid, signal.SIGKILL)
        except Exception:  # noqa: BLE001
            pass
        p.communicate()
        return None, "timeout"
    finally:
        if p is not None and p.poll() is None:
            try:
                os.killpg(p.pid, signal.SIGKILL)
            except Exception:  # noqa: BLE001
                pass
        shutil.rmtree(cwd, ignore_errors=True)
    try:
        out = json.loads(stdout.strip().splitlines()[-1])
    except Exception:  # noqa: BLE001
        return None, "no output (killed or crashed)"
    if "compile_error" in out:
        return None, out["compile_error"]
    return out["results"], None


def check(code, task):
    """Run on train + test inputs of the (augmented) task. verified = every train output reproduced exactly and every
    test output is a valid grid."""
    grids = [p["input"] for p in task["train"]] + [p["input"] for p in task["test"]]
    res, err = run_program(code, grids)
    n = len(task["train"])
    if res is None:
        return {"verified": False, "error": err, "train_pass": 0, "train_n": n, "tests": None}
    passes = [r.get("ok") == p["output"] for r, p in zip(res[:n], task["train"])]
    tests = [r.get("ok") if L.valid_grid(r.get("ok")) else None for r in res[n:]]
    return {"verified": all(passes) and all(t is not None for t in tests), "error": None, "train_pass": sum(passes),
            "train_n": n, "tests": tests, "train_results": res[:n]}


def feedback(task, chk):
    """One round of execution feedback for a failed program."""
    if chk.get("error"):
        return textwrap.shorten(f"Your code failed to run: {chk['error']}", 600) + \
            "\nFix it. Give the complete corrected code in one ```python block at the end."
    lines = [f"Your program reproduces {chk['train_pass']} of {chk['train_n']} example outputs. Mismatches:"]
    for i, (r, p) in enumerate(zip(chk["train_results"], task["train"]), 1):
        if r.get("ok") == p["output"]:
            continue
        if "err" in r:
            lines.append(f"Example {i}: error {r['err'][:200]}")
            continue
        got = r.get("ok")
        if not L.valid_grid(got) or len(got) * len(got[0]) > 400 or len(p["output"]) * len(p["output"][0]) > 400:
            shape = f"{len(got)}x{len(got[0]) if got and isinstance(got[0], list) else '?'}" if isinstance(got, list) else "?"
            lines.append(f"Example {i}: expected shape {len(p['output'])}x{len(p['output'][0])}, got {shape}")
        else:
            lines.append(f"Example {i}: expected\n{L.grid_text(p['output'])}\ngot\n{L.grid_text(got)}")
    lines.append("Find the rule that fits every example, then give the complete corrected code in one ```python block "
                 "at the end.")
    return "\n".join(lines)


In [ ]:
%%writefile v13_merge.py
"""v13 fill rule (pre-registered, HANDOFF 2026-10-13): per test output, the answer of the verified programs (majority,
ties -> first verified) becomes attempt_1 and NVARC pass A's top-1 attempt_2; when the program answer already is one of
NVARC's two attempts the entry is left unchanged; when NVARC has no real grid, attempt_2 is the next distinct program
answer. Pure functions + atomic write; no third-party imports."""
import json
import os

PLACEHOLDER = [[0]]


def valid_grid(g):
    return (isinstance(g, list) and 0 < len(g) <= 30 and all(isinstance(r, list) for r in g)
            and 0 < len(g[0]) <= 30 and all(len(r) == len(g[0]) for r in g)
            and all(type(v) is int and 0 <= v <= 9 for r in g for v in r))


def ranked_program_answers(answer_lists, i):
    """Distinct program answers for test output i, most-supported first, ties by first appearance."""
    order, count = [], {}
    for answers in answer_lists:
        try:
            g = answers[i]
        except (IndexError, TypeError):
            continue
        if not valid_grid(g):
            continue
        key = json.dumps(g)
        if key not in count:
            order.append(key)
            count[key] = 0
        count[key] += 1
    order.sort(key=lambda k: -count[k])  # stable: ties keep first appearance
    return [json.loads(k) for k in order]


def fill_entry(base_entry, programs):
    """base_entry: {'attempt_1', 'attempt_2'} from NVARC (may be placeholders); programs: ranked distinct grids."""
    if not programs:
        return dict(base_entry)
    nv = [base_entry.get(a) for a in ("attempt_1", "attempt_2")]
    nv = [g for g in nv if valid_grid(g) and g != PLACEHOLDER]
    best = programs[0]
    if best in nv:
        return dict(base_entry)
    second = nv[0] if nv else next((g for g in programs[1:] if g != best), None)
    if second is None:
        second = base_entry.get("attempt_1") if valid_grid(base_entry.get("attempt_1")) else PLACEHOLDER
    return {"attempt_1": best, "attempt_2": second}


def merge(test, base, results):
    """test: challenges dict; base: NVARC submission (same keys); results: task -> list of verified answer lists
    (one grid per test output, original frame). Returns (merged, n_changed_outputs)."""
    merged, changed = {}, 0
    for key, task in test.items():
        entries = []
        for i in range(len(task["test"])):
            try:
                b = dict(base[key][i])
            except Exception:  # noqa: BLE001  (missing/odd base entry: placeholders)
                b = {"attempt_1": PLACEHOLDER, "attempt_2": PLACEHOLDER}
            for a in ("attempt_1", "attempt_2"):
                if not valid_grid(b.get(a)):
                    b[a] = PLACEHOLDER
            e = fill_entry(b, ranked_program_answers(results.get(key, []), i))
            changed += e != b
            entries.append(e)
        merged[key] = entries
    return merged, changed


def validate(test, submission):
    if set(submission) != set(test):
        raise ValueError("submission keys differ from test keys")
    for k, task in test.items():
        entries = submission[k]
        if not isinstance(entries, list) or len(entries) != len(task["test"]):
            raise ValueError(f"{k}: wrong number of test entries")
        for e in entries:
            if not isinstance(e, dict) or set(e) != {"attempt_1", "attempt_2"}:
                raise ValueError(f"{k}: attempts missing")
            if not all(valid_grid(g) for g in e.values()):
                raise ValueError(f"{k}: invalid grid")
    return True


def atomic_json(path, value):
    tmp = str(path) + ".tmp"
    with open(tmp, "w") as f:
        json.dump(value, f)
    os.replace(tmp, path)


def load_results(path):
    """task -> list of verified answer lists, from the driver's programs jsonl (verified rows only)."""
    res = {}
    try:
        with open(path) as f:
            for line in f:
                try:
                    r = json.loads(line)
                except ValueError:
                    continue
                if r.get("verified") and isinstance(r.get("answers"), list):
                    res.setdefault(r["task"], []).append(r["answers"])
    except OSError:
        pass
    return res


def finalize(work_dir="/kaggle/working", test_path=None):
    """Last notebook cell (rerun): rebuild submission.json from the saved NVARC base + verified programs; on any
    problem keep a valid submission (current file if valid, else the base). Never raises."""
    sub = os.path.join(work_dir, "submission.json")
    base_path = os.path.join(work_dir, "v13-base-submission.json")
    record = {"finalize": None}
    try:
        with open(test_path) as f:
            test = json.load(f)
        if not os.path.exists(base_path):
            record["finalize"] = "no base (LLM phase did not start); submission.json left as assembled"
            return record
        with open(base_path) as f:
            base = json.load(f)
        merged, changed = merge(test, base, load_results(os.path.join(work_dir, "v13-programs.jsonl")))
        validate(test, merged)
        atomic_json(sub, merged)
        record.update(finalize="merged", changed_outputs=changed)
    except Exception as e:  # noqa: BLE001
        record["finalize"] = f"error {type(e).__name__}: {e}"[:300]
        try:
            with open(test_path) as f:
                test = json.load(f)
            with open(sub) as f:
                validate(test, json.load(f))
            record["kept"] = "current submission.json (valid)"
        except Exception:  # noqa: BLE001
            try:
                with open(base_path) as f:
                    base = json.load(f)
                validate(test, base)
                atomic_json(sub, base)
                record["kept"] = "base"
            except Exception as e2:  # noqa: BLE001
                record["kept"] = f"nothing valid: {type(e2).__name__}"
    finally:
        print(json.dumps({"msg": "v13_finalize", **record}), flush=True)
    return record


In [ ]:
%%writefile v13_llm.py
"""v13 LLM phase (runs after NVARC pass A has been assembled into submission.json): serve Qwen3.8-27B-FP8 with vLLM
0.19.0 from a Python 3.12 venv (python-build-standalone + the cp312 wheelhouse; the NVARC Python 3.11 stack is never
touched) and run the program-synthesis channel: write `transform(grid)`, verify it on every train pair of an augmented
copy of the task, and when it passes put its test answer into attempt_1 (v13_merge fill rule). Every verified program
rewrites the submission atomically, so a hard stop at any moment leaves a valid file.

Any failure before the first job (time, inputs missing, GPUs busy, install, server) skips the phase and keeps NVARC's
submission. Rerun: tasks from the competition test file only, unsure-first by NVARC pass-A confidence. Commit: the
4 NVARC smoke tasks + the 30 public-eval keys30 tasks, scored against the evaluation solutions (dev measurement).
  python v13_llm.py --end-time T   (T = epoch seconds by which this phase must have returned)"""
import argparse
import glob
import hashlib
import json
import os
import re
import signal
import socket
import subprocess
import sys
import threading
import time
import zlib
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
import arc_llm as L  # noqa: E402
import progsynth as P  # noqa: E402
import v13_merge as M  # noqa: E402

T0 = time.time()
A = None
S = {"version": "v13-llm-v3b", "t0": T0, "phases": {}, "skipped": None}
LOCK = threading.Lock()
RESULTS = {}           # task -> list of verified answer lists (original frame)
STOP = threading.Event()
ACTIVE = {}          # open streaming responses; closed at the abort time (a preempted stream sends nothing)
SERVER = [None]        # the server process, for the last-resort kill after the abort time
PORT = int(os.environ.get("V13_PORT", "8000"))  # test seam; production 8000
MAX_LEN = 131072
PY_TGZ = "cpython-3.12.14+20260924-x86_64-unknown-linux-gnu-install_only.tar.gz"
PY_SHA = "5eae8cf79dd47fc2496a4fccc892936be831ce7a84d984b2299dfb1cdb592682"
PYPI = {  # sha256 of the official PyPI wheels (pypi.org JSON API, 2026-10-12; same pins as P0/P1)
    "vllm-0.19.0-cp38-abi3-manylinux_2_31_x86_64.whl":
        "2d0e5fae45367bdbf111fcad68f4c0f8fdddd2f2fb643e52f0f2daebef7b41cf",
    "torch-2.10.0-3-cp312-cp312-manylinux_2_28_x86_64.whl":
        "98c01b8bb5e3240426dcde1446eed6f40c778091c8544767ef1168fc663a05a6",
}
CONFIDENT = 5.0        # arc_v6_config.PASS_B_CONFIDENT: kgmon top-1 score counted as sure (v9)
MAX_REPS = 6           # augmentation repetitions per task; rep r >= 1 only for tasks without a verified program
THINK_END = [None]     # v3: the template's end-of-thinking token id ("</think>" via /tokenize at server start)
SUFFIX_IDS = [None]    # v3: FORCE_TEXT token ids, tokenized once at server start (token_id_parity)
FORCES = {"tried": 0, "ok": 0}
BREAKER_K = 3          # v3: the first K forcings all failing turns forcing off for jobs not yet started


def el():
    return round(time.time() - T0, 1)


def log(msg):
    print(f"[v13 {el():8.1f}s] {msg}", flush=True)


def save():
    with LOCK:
        text = json.dumps(S, default=str)
        tmp = os.path.join(A.work, "v13-llm.json.tmp")
        with open(tmp, "w") as f:
            f.write(text)
        os.replace(tmp, os.path.join(A.work, "v13-llm.json"))


def append(name, row):
    with LOCK, open(os.path.join(A.work, name), "a") as f:
        f.write(json.dumps(row, default=str) + "\n")


def run(cmd, timeout=None, **kw):
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, **kw)
        return p.returncode, p.stdout + p.stderr
    except Exception as e:  # noqa: BLE001
        return None, repr(e)


def sha_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""):
            h.update(b)
    return h.hexdigest()


def find(pattern):
    return sorted(glob.glob(os.path.join(A.input_root, "**", pattern), recursive=True))


class Skip(Exception):
    pass


def budget(seconds):
    """Timeout for one setup step: never past the setup deadline (server must be ready by then); skip when < 30 s."""
    left = A.setup_deadline - time.time()
    if left < 30:
        raise Skip(f"setup time exhausted ({round(left)}s left)")
    return min(seconds, left)


def venv_env():
    """Environment for every Python 3.12 venv process (pip, checks, server): nothing from the NVARC 3.11 stack leaks in
    (PYTHONPATH/HOME, user site, PIP_*), and the CUDA toolkit dirs do not shadow the torch wheel's bundled CUDA libs."""
    drop = ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV", "PYTHONSTARTUP", "PYTHONUSERBASE", "CONDA_PREFIX")
    env = {k: v for k, v in os.environ.items() if k not in drop and not k.startswith("PIP_")}
    env["PYTHONNOUSERSITE"] = "1"
    parts = [x for x in env.get("LD_LIBRARY_PATH", "").split(":") if x and not x.startswith("/usr/local/cuda")]
    libcuda = "/usr/local/nvidia/lib64/libcuda.so"
    if os.path.exists(libcuda):  # same libcuda pin as P0/P1 (public Apache-2.0 Qwen3.8 notebook)
        parts = ["/usr/local/nvidia/lib64"] + [x for x in parts if x != "/usr/local/nvidia/lib64"]
        env["LD_PRELOAD"] = libcuda
    env["LD_LIBRARY_PATH"] = ":".join(parts)
    return env


# --- 1. inputs, GPUs, install ------------------------------------------------------------------------------------------

def phase_inputs():
    ph = S["phases"]["inputs"] = {}
    wheels = find("vllm-0.19.0-*.whl")
    tgz = find(PY_TGZ)
    model = [os.path.dirname(p) for p in find("layers-0.safetensors")]
    ph.update(wheels=wheels[:1], py_tgz=tgz[:1], model=model[:2])
    if not (wheels and tgz and len(model) == 1):
        raise Skip(f"inputs missing: wheels={len(wheels)} py={len(tgz)} model={len(model)}")
    return Path(wheels[0]).parent, tgz[0], Path(model[0])


def gpu_used():
    rc, out = run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], timeout=60)
    vals = [int(x) for x in out.split() if x.strip().isdigit()] if rc == 0 else []
    return vals


def phase_gpus():
    """NVARC workers must be gone: all GPUs near-empty within 5 min, else kill orphan compute processes, else skip."""
    ph = S["phases"]["gpus"] = {}
    for attempt in range(2):
        end = min(time.time() + (300 if attempt == 0 else 120), A.setup_deadline)
        while time.time() < end:
            used = gpu_used()
            if A.fake_server or (used and all(u < 1500 for u in used)):
                ph.update(free_after=el(), used=used, attempt=attempt)
                return
            time.sleep(10)
        rc, out = run(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"], timeout=60)
        pids = [int(x) for x in out.split() if x.strip().isdigit()] if rc == 0 else []
        mine = ancestors()
        killed = []
        for pid in pids:  # only NVARC leftovers: container PIDs whose command line is a starter/multiprocessing child
            try:
                cmd = Path(f"/proc/{pid}/cmdline").read_bytes().replace(b"\0", b" ").decode(errors="ignore")
            except OSError:
                continue
            if pid not in mine and ("multiprocessing" in cmd or "starter.py" in cmd):
                try:
                    os.kill(pid, signal.SIGKILL)
                    killed.append(pid)
                except Exception:  # noqa: BLE001
                    pass
        rc2, _ = run(["pkill", "-9", "-f", "multiprocessing.spawn"], timeout=30)  # orphan NVARC workers/manager
        ph.setdefault("killed", []).append({"pids": killed, "pkill_rc": rc2, "used": gpu_used()})
        save()
    raise Skip(f"GPUs not released: {gpu_used()}")


def ancestors():
    pids, pid = set(), os.getpid()
    while pid > 1 and pid not in pids:
        pids.add(pid)
        try:
            pid = int(Path(f"/proc/{pid}/stat").read_text().rsplit(")", 1)[1].split()[1])
        except Exception:  # noqa: BLE001
            break
    return pids | {os.getppid()}


def verify_wheels(wheels):
    ph = S["phases"]["wheels"] = {}
    sums = {}
    for line in (wheels / "SHA256SUMS").read_text().splitlines():
        parts = line.split()
        if len(parts) == 2:
            sums[parts[1].lstrip("*")] = parts[0]
    files = sorted(p.name for p in wheels.glob("*.whl"))
    bad = [n for n in files if sums.get(n) != sha_file(wheels / n)]
    pins = {n: (sums.get(n) == h and (wheels / n).exists()) for n, h in PYPI.items()}
    ph.update(n_wheels=len(files), mismatched=bad, pypi_pins=pins)
    if bad or not all(pins.values()):
        raise Skip("wheel hash check failed")


def model_hash(model, out):
    """Commit only, informational (P1 already verified this pinned mirror; train-pair verification guards answers):
    sha256 of every weight file vs the HF LFS list. Never blocks the phase."""
    from concurrent.futures import ThreadPoolExecutor
    try:
        ref = json.loads((HERE / "qwen38_fp8_sha256.json").read_text())["files"]
        with ThreadPoolExecutor(max_workers=4) as pool:
            futs = {n: pool.submit(sha_file, model / n) for n in ref if (model / n).exists()}
            bad = [n for n, f in futs.items() if f.result() != ref[n][1] or (model / n).stat().st_size != ref[n][0]]
        res = dict(files=len(ref), missing=[n for n in ref if n not in futs], mismatched=bad, done=el())
    except Exception as e:  # noqa: BLE001
        res = {"error": repr(e)[:200]}
    with LOCK:
        out.update(res)


def phase_install(wheels, tgz):
    ph = S["phases"]["install"] = {}
    t = time.time()
    if sha_file(tgz) != PY_SHA:
        raise Skip("python tarball sha256 mismatch")
    root = Path(A.tmp)
    root.mkdir(parents=True, exist_ok=True)
    rc, out = run(["tar", "xzf", tgz, "-C", str(root)], timeout=budget(900))
    if rc != 0:
        raise Skip("python extract failed: " + out[-300:])
    venv = root / "venv"
    rc, out = run([str(root / "python/bin/python3.12"), "-m", "venv", str(venv)], timeout=budget(900), env=venv_env())
    if rc != 0:
        raise Skip("venv failed: " + out[-300:])
    vpy = str(venv / "bin/python")
    verify_wheels(wheels)
    rc, out = run([vpy, "-m", "pip", "install", "--no-index", "--find-links", str(wheels), "--requirement",
                   str(wheels / "requirements.lock")], timeout=budget(2400), env=venv_env())
    ph.update(pip_rc=rc, pip_tail=out[-1500:], pip_seconds=round(time.time() - t))
    if rc != 0:
        raise Skip("pip install failed")
    rc, out = run([vpy, "-c", "import json, torch, vllm, transformers; print(json.dumps({'torch': torch.__version__, "
                   "'cuda_ok': torch.cuda.is_available(), 'gpus': torch.cuda.device_count(), "
                   "'vllm': vllm.__version__, 'transformers': transformers.__version__}))"], timeout=budget(900),
                  env=venv_env())
    ph["import"] = out.strip().splitlines()[-1] if out.strip() else out
    if rc != 0 or '"cuda_ok": true' not in ph["import"]:
        raise Skip("venv torch/vllm import or CUDA check failed")
    rc2, _ = run([vpy, "-c", "import flash_attn"], timeout=budget(300), env=venv_env())
    ph["flash_attn"] = rc2 == 0
    if rc2 != 0:  # P0/P1 fix: vLLM forwards layer_idx to an attention impl that rejects it without flash-attn
        rc3, vroot = run([vpy, "-c", "import os, vllm; print(os.path.dirname(vllm.__file__))"], timeout=budget(300),
                         env=venv_env())
        vroot = vroot.strip().splitlines()[-1] if rc3 == 0 and vroot.strip() else ""
        patched, reverted = [], []
        for f in sorted(glob.glob(vroot + "/**/attention/*.py", recursive=True)) if vroot else []:
            text = Path(f).read_text()
            if "self.impl = impl_cls(" not in text or "extra_impl_args.pop('layer_idx', None)" in text:
                continue
            new = re.sub(r"^(\s*)self\.impl = impl_cls\(",
                         r"\1extra_impl_args.pop('layer_idx', None)\n\1self.impl = impl_cls(", text, count=1, flags=re.M)
            Path(f).write_text(new)
            if run([vpy, "-m", "py_compile", f], timeout=budget(120), env=venv_env())[0] != 0 \
                    or "extra_impl_args" not in text:
                Path(f).write_text(text)
                reverted.append(f)
            else:
                patched.append(f)
        ph.update(attention_patch=patched, attention_patch_reverted=reverted)
    ph["seconds"] = round(time.time() - t)
    save()
    return vpy


# --- 2. server --------------------------------------------------------------------------------------------------------

def server_env():
    return dict(venv_env(), PYTORCH_ALLOC_CONF="expandable_segments:True",
                VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS="1", HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
                VLLM_NO_USAGE_STATS="1", DO_NOT_TRACK="1", VLLM_LOGGING_LEVEL="INFO")  # backend: --attention-backend


def http(method, path, body=None, timeout=30):
    import requests
    r = requests.request(method, f"http://127.0.0.1:{PORT}{path}", json=body, timeout=timeout)
    return r.status_code, r.text


def wait_ready(proc, deadline):
    while time.time() < deadline:
        if proc.poll() is not None:
            return False
        try:
            if http("GET", "/v1/models", timeout=5)[0] == 200:
                return True
        except Exception:  # noqa: BLE001
            pass
        time.sleep(5)
    return False


def kill(proc):
    if proc is None or proc.poll() is not None:
        return
    try:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        proc.wait(timeout=60)
    except Exception:  # noqa: BLE001
        try:
            os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
        except Exception:  # noqa: BLE001
            pass


def start_server(vpy, model, mtp, eager):
    if A.fake_server:  # test seam: a stub OpenAI-compatible server
        cmd = [sys.executable, A.fake_server, "--port", str(PORT)]
    else:
        cmd = [vpy, "-m", "vllm.entrypoints.openai.api_server", "--model", str(model), "--served-model-name", "m",
               "--host", "127.0.0.1", "--port", str(PORT), "--tensor-parallel-size", "4",
               "--gpu-memory-utilization", "0.93", "--max-model-len", str(MAX_LEN),
               "--max-num-seqs", str(max(48, A.conc)), "--kv-cache-dtype", "fp8", "--attention-backend",
               "TRITON_ATTN", "--reasoning-parser", "qwen3", "--trust-remote-code",
               "--limit-mm-per-prompt", json.dumps({"image": 0, "video": 0})]
        if mtp:
            cmd += ["--speculative-config", json.dumps({"method": "mtp", "num_speculative_tokens": mtp})]
        if eager:
            cmd += ["--enforce-eager"]
    path = os.path.join(A.work, f"v13-server-mtp{mtp}{'-eager' if eager else ''}.log")
    proc = subprocess.Popen(cmd, stdout=open(path, "w"), stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                            text=True, start_new_session=True, env=server_env())
    return proc, path


def smoke():
    body = {"model": "m", "messages": [{"role": "user", "content": "Reply with the single word OK."}],
            "max_tokens": 32, "temperature": 0, "chat_template_kwargs": {"enable_thinking": False}}
    code, text = http("POST", "/v1/chat/completions", body, timeout=300)
    return code == 200, text[-300:]


def tokenize(text):
    code, body = http("POST", "/tokenize", {"model": "m", "prompt": text, "add_special_tokens": False}, timeout=60)
    return json.loads(body).get("tokens") if code == 200 else None


def continuation_prompt(pids, oids):
    """(prompt ids, in_answer) for a forcing: when the end-of-thinking id is among the output ids the model already left
    its thinking -> continue its answer as is; else append the cached forced end of thinking that opens a code block."""
    in_answer = THINK_END[0] is not None and THINK_END[0] in oids
    return pids + oids + ([] if in_answer else list(SUFFIX_IDS[0] or [])), in_answer


def sampling(seed):
    return {"temperature": 1.0, "top_p": 0.95, "top_k": 20, "min_p": 0.0, "seed": seed}


def token_id_parity():
    """v3 forcing precondition, checked once at server start on the real forcing path: (1) "</think>" is one token and
    FORCE_TEXT (tokenized once, cached) contains it exactly once; (2) a thinking request with the jobs' template kwargs
    and sampling, cut by max_tokens, streams prompt ids and exactly usage.completion_tokens output ids; (3) a
    /v1/completions continuation on those ids (+ suffix) returns text. Any failure turns forcing off (v2 behaviour)."""
    res = {"ok": False}
    try:
        end, suffix = tokenize("</think>"), tokenize(FORCE_TEXT)
        res.update(think_end=end, suffix_ids=len(suffix or []))
        if not end or len(end) != 1 or not suffix or suffix.count(end[0]) != 1:
            res["error"] = "end-of-thinking token check failed"
            return res
        THINK_END[0], SUFFIX_IDS[0] = end[0], suffix
        r = _stream("/v1/chat/completions", {
            "model": "m", "messages": [{"role": "user", "content": "Think it through carefully: for which integers "
                                        "0 <= n < 60 is n*n + n + 41 composite? Explain each case."}],
            "max_tokens": 300, **sampling(1), "stream": True, "return_token_ids": True,
            "stream_options": {"include_usage": True, "continuous_usage_stats": True},
            "chat_template_kwargs": {"reasoning_effort": A.effort, "enable_thinking": True}},
            {"state": "running", "t_start": el()})
        pids, oids = r.get("prompt_token_ids") or [], r["token_ids"]
        res.update(state=r["state"], finish=r.get("finish_reason"), prompt_ids=len(pids), out_ids=len(oids),
                   completion_tokens=r.get("completion_tokens"), error=r.get("error"))
        if not (r["state"] == "done" and r.get("finish_reason") == "length" and pids and oids
                and len(oids) == r.get("completion_tokens")):
            return res
        prompt, in_answer = continuation_prompt(pids, oids)
        c = _stream("/v1/completions", {"model": "m", "prompt": prompt, "max_tokens": 32, **sampling(2), "stream": True,
                                        "skip_special_tokens": True,
                                        "stream_options": {"include_usage": True, "continuous_usage_stats": True}},
                    {"state": "running", "t_start": el()})
        res.update(in_answer=in_answer, cont_state=c["state"], cont_tokens=c.get("completion_tokens"),
                   cont_text=c["text"][:80], cont_error=c.get("error"), cont_t_first=c.get("t_first"))
        res["ok"] = c["state"] == "done" and bool(c["text"].strip()) and (c.get("completion_tokens") or 0) > 0
    except Exception as e:  # noqa: BLE001
        res["error"] = repr(e)[-300:]
    return res


def phase_server(vpy, model, deadline):
    ph = S["phases"]["server"] = {"attempts": []}
    for mtp, eager in ((2, False), (0, False), (0, True)):
        if time.time() > deadline:
            break
        t = time.time()
        proc, path = start_server(vpy, model, mtp, eager)
        ready = wait_ready(proc, min(t + 25 * 60, deadline))
        att = {"mtp": mtp, "eager": eager, "ready": ready, "startup_seconds": round(time.time() - t)}
        if ready:
            ok, resp = smoke()
            att.update(smoke_ok=ok, smoke=resp)
            text = Path(path).read_text(errors="ignore")
            kv = re.findall(r"GPU KV cache size:\s*([\d,]+)\s*tokens", text)
            att["kv_tokens"] = int(kv[-1].replace(",", "")) if kv else None
            if ok:
                ph["attempts"].append(att)
                ph["chosen"] = att
                save()
                log(f"server ready mtp={mtp} eager={eager} in {att['startup_seconds']}s kv={att['kv_tokens']}")
                return proc
        att["log_tail"] = Path(path).read_text(errors="ignore")[-2500:] if os.path.exists(path) else ""
        ph["attempts"].append(att)
        save()
        kill(proc)
        time.sleep(10)
    raise Skip("no server configuration started")


# --- 3. jobs ------------------------------------------------------------------------------------------------------------

_CODE = re.compile(r"```(?:python|py)?\s*\n(.*?)```", re.S)


def code_blocks(text):
    return [b for b in _CODE.findall(text or "") if "def transform" in b]


def prompt_tokens(messages):
    try:
        code, text = http("POST", "/tokenize", {"model": "m", "messages": messages, "chat_template_kwargs":
                                                 {"reasoning_effort": A.effort, "enable_thinking": True}}, timeout=60)
        if code == 200 and json.loads(text).get("count"):
            return json.loads(text)["count"]
    except Exception:  # noqa: BLE001
        pass
    return sum(len(m["content"]) for m in messages) + 300  # characters: an upper bound for these digit grids


def _stream(path, body, out):
    """POST a streaming request and collect it into `out` (content, reasoning, text, token ids, usage, finish). The
    abort time and STOP cut the stream; the socket is registered so abort_streams() can shut it down."""
    import requests
    content, reasoning, text, ids, usage, finish = [], [], [], [], {}, None
    resp = None
    try:
        with requests.post(f"http://127.0.0.1:{PORT}{path}", json=body, stream=True, timeout=(30, 900)) as r:
            if r.status_code != 200:  # still falls through to the final update (content/reasoning/text = "")
                out.update(state="http_error", error=r.text[-300:])
            else:
                resp = r
                with LOCK:
                    ACTIVE[id(r)] = r
                if STOP.is_set():
                    r.close()
            for raw in (r.iter_lines(chunk_size=None) if resp is not None else ()):  # as bytes arrive (no 512-byte buffer); split on \r/\n only
                raw = raw.decode("utf-8", "replace")
                if STOP.is_set() or time.time() > A.abort_at:
                    out["state"] = "aborted"
                    break
                if not raw or not raw.startswith("data: ") or raw[6:].strip() == "[DONE]":
                    continue
                ev = json.loads(raw[6:])
                if "t_first" not in out:  # time to first event (continuations: queueing + ~75k-token prefill)
                    out["t_first"] = el()
                usage = ev.get("usage") or usage
                if ev.get("prompt_token_ids") and "prompt_token_ids" not in out:
                    out["prompt_token_ids"] = ev["prompt_token_ids"]
                for ch in ev.get("choices") or []:
                    d = ch.get("delta") or {}
                    if d.get("content"):
                        content.append(d["content"])
                    rs = d.get("reasoning_content") or d.get("reasoning")
                    if rs:
                        reasoning.append(rs)
                    if ch.get("text"):
                        text.append(ch["text"])
                    if ch.get("prompt_token_ids") and "prompt_token_ids" not in out:
                        out["prompt_token_ids"] = ch["prompt_token_ids"]
                    if ch.get("token_ids"):
                        ids.extend(ch["token_ids"])
                    finish = ch.get("finish_reason") or finish
    except Exception as e:  # noqa: BLE001
        out.update(state="aborted" if STOP.is_set() else "exception", error=repr(e)[-300:])
    finally:
        if resp is not None:
            with LOCK:
                ACTIVE.pop(id(resp), None)
    if STOP.is_set() and out["state"] == "running" and not finish:
        out["state"] = "aborted"
    out.update(t_end=el(), completion_tokens=usage.get("completion_tokens"), prompt_tokens=usage.get("prompt_tokens"),
               finish_reason=finish, content="".join(content), reasoning="".join(reasoning), text="".join(text),
               token_ids=ids)
    if out["state"] == "running":
        out["state"] = "done" if finish else "incomplete"
    return out


def chat(messages, cap, seed):
    body = {"model": "m", "messages": messages, "max_tokens": cap, "temperature": 1.0, "top_p": 0.95, "top_k": 20,
            "min_p": 0.0, "seed": seed, "stream": True, "stream_options": {"include_usage": True, "continuous_usage_stats": True},
            "chat_template_kwargs": {"reasoning_effort": A.effort, "enable_thinking": True}}
    if A.force_at:  # budget forcing needs the exact prompt + output token ids for the continuation
        body["return_token_ids"] = True
    return _stream("/v1/chat/completions", body, {"state": "running", "t_start": el(), "max_tokens": cap})


FORCE_TEXT = ("\n\nMy thinking budget is used up. I will commit to the most likely rule found so far and write the "
              "complete program now.\n</think>\n\n```python\n")


def force_continue(r, seed, room):
    """Budget forcing (v3): the first request stopped at the forcing point (`length`). Continue the SAME sequence via
    /v1/completions on token ids (prompt ids + generated ids + a forced end of thinking that opens a code block; when
    the model had already left its thinking, just continue its answer) for up to `room` tokens. Token ids, not a
    re-rendered chat template, so the model sees exactly its own sequence; MTP stays on (vLLM 0.19 ignores the native
    thinking_token_budget under speculative decoding). Returns (answer_text, info) or (None, info)."""
    info = {"forced": True}
    pids, oids = r.get("prompt_token_ids") or [], r.get("token_ids") or []
    if not pids or not oids or (r.get("completion_tokens") and len(oids) != r["completion_tokens"]):
        info["force_error"] = f"token ids missing/incomplete: prompt {len(pids)}, out {len(oids)}, " \
                              f"usage {r.get('completion_tokens')}"
        return None, info
    if not SUFFIX_IDS[0]:
        info["force_error"] = "no cached forced suffix (parity not passed)"
        return None, info
    prompt, in_answer = continuation_prompt(pids, oids)
    room = min(room, MAX_LEN - len(prompt) - 16)
    info.update(in_answer=in_answer, cont_prompt_tokens=len(prompt), cont_max_tokens=room)
    if room < 256:
        info["force_error"] = "no room"
        return None, info
    body = {"model": "m", "prompt": prompt, "max_tokens": room, **sampling(seed), "stream": True,
            "skip_special_tokens": True, "stream_options": {"include_usage": True, "continuous_usage_stats": True}}
    c = _stream("/v1/completions", body, {"state": "running", "t_start": el()})
    info.update({f"cont_{k}": c.get(k) for k in ("state", "finish_reason", "completion_tokens", "t_start", "t_first",
                                                    "t_end", "error")})
    if not (c.get("state") == "done" and (c.get("text") or "").strip() and (c.get("completion_tokens") or 0) > 0):
        info["force_error"] = f"continuation {c.get('state')}: {(c.get('error') or 'no text')[-200:]}"
        if not (c.get("text") or "").strip():
            return None, info
    answer = (r["content"] + c["text"]) if in_answer else ("```python\n" + c["text"])
    if answer.count("```") % 2:  # cut at the length limit / abort inside the code block: close it so it can be tried
        answer += "\n```"
    return answer, info


def suspicious(code, task, tests):
    """Cheap guards against programs that pass the train pairs by memorising them (API review 20261013T150520Z #12):
    a train output (>= 9 cells, >= 2 colours) written into the source, a test answer that copies a train output
    (>= 9 cells), or an identity answer on test when no train pair is identity. Returns a reason or None."""
    digits = re.sub(r"\D", "", code)
    outs = [p["output"] for p in task["train"]]
    for o in outs:
        flat = "".join(str(v) for r in o for v in r)
        if len(flat) >= 9 and len(set(flat)) >= 2 and flat in digits:
            return "train output hard-coded"
    for q, t in zip(task["test"], tests):
        if sum(len(r) for r in t) >= 9 and any(t == o for o in outs):
            return "test answer copies a train output"
        if t == q["input"] and not any(p["input"] == p["output"] for p in task["train"]):
            return "identity on test only"
    return None


def run_job(task_id, rep):
    raw = TASKS[task_id]
    aug = L.augmentation(task_id, salt=f"v13-r{rep}")
    task = L.augment_task(raw, aug)
    msgs = [{"role": "user", "content": P.build_prompt(task)}]
    n_prompt = prompt_tokens(msgs)
    cap = min(A.cap, MAX_LEN - n_prompt - 512)
    row = {"task": task_id, "rep": rep, "prompt_tokens_est": n_prompt, "cap": cap}
    if cap < 2000:
        row.update(state="prompt_too_long", verified=False)
        append("v13-programs.jsonl", row)
        return row
    seed = zlib.crc32(f"v13:{task_id}:{rep}".encode()) & 0x7FFFFFFF
    r = chat(msgs, min(cap, A.force_at) if A.force_at else cap, seed)
    row.update({k: r.get(k) for k in ("state", "finish_reason", "completion_tokens", "prompt_tokens", "t_start",
                                      "t_end", "error")})
    forced = None
    if (A.force_at and r.get("state") == "done" and r.get("finish_reason") == "length" and not code_blocks(r["content"])
            and not STOP.is_set() and time.time() < A.stop_new_at):  # prefill + a program need minutes, not 60 s
        room = min(A.answer_tokens, cap - (r.get("completion_tokens") or A.force_at))
        forced, info = force_continue(r, seed ^ 0x2545F491, room)
        row.update(info)
        breaker(info)
    # candidates: the final answer's code first; then the forced continuation's code (v3); then (truncated, aborted
    # or unverified) the last code drafts of the reasoning — train-pair verification guards precision
    cands = [("content", c) for c in code_blocks(r["content"])[-1:]]
    cands += [("forced", c) for c in code_blocks(forced)[-1:]]
    cands += [("reasoning", c) for c in code_blocks(r["reasoning"])[-2:][::-1]]
    row.update(n_content_code=len(code_blocks(r["content"])), n_reasoning_code=len(code_blocks(r["reasoning"])),
               reasoning_len=len(r["reasoning"]), content_len=len(r["content"]), verified=False, tried=[])
    if r.get("finish_reason") != "stop":  # what the model was doing at the cap / abort (input to cap & effort choices)
        row["reasoning_tail"] = r["reasoning"][-2000:]
    seen = set()
    for source, code in cands:
        if code in seen:
            continue
        seen.add(code)
        chk = P.check(code, task)
        why = suspicious(code, task, chk["tests"]) if chk["verified"] else None
        row["tried"].append({"source": source, "verified": chk["verified"], "train_pass": chk["train_pass"],
                             "train_n": chk["train_n"], "error": (chk.get("error") or "")[:200], "rejected": why})
        if chk["verified"] and why is None:
            answers = [L.invert_aug(t, aug) for t in chk["tests"]]
            row.update(verified=True, source=source, answers=answers, code_tail=code[-1500:])
            if SOLUTIONS.get(task_id):
                row["correct"] = [a == s for a, s in zip(answers, SOLUTIONS[task_id])]
            break
    append("v13-programs.jsonl", row)  # the merge reads verified rows from this file
    if row["verified"]:
        with LOCK:
            RESULTS.setdefault(task_id, []).append(row["answers"])
        write_submission()
    return row


def breaker(info):
    """v3 run-time circuit breaker (the only guard on the hidden rerun): when the first BREAKER_K forcings all failed,
    jobs not yet started run as v2 (cap, no forcing)."""
    tripped = False
    with LOCK:
        FORCES["tried"] += 1
        FORCES["ok"] += not info.get("force_error")
        if A.force_at and FORCES["tried"] >= BREAKER_K and not FORCES["ok"]:
            A.force_at = 0
            S["force_disabled"] = f"circuit breaker: the first {FORCES['tried']} forcings all failed"
            tripped = True
    if tripped:
        log(S["force_disabled"])


def write_submission():
    """Atomic rewrite from the NVARC base + every verified program so far (rerun: submission.json); one lock section,
    so an older snapshot can never land after a newer one."""
    with LOCK:
        merged, changed = M.merge(TEST, BASE, {k: list(v) for k, v in RESULTS.items()})
        M.validate(TEST, merged)
        M.atomic_json(os.path.join(A.work, OUT_NAME), merged)
        S["changed_outputs"] = changed


def job_order():
    """Rerun: NVARC pass-A confidence ascending (no candidate = -inf first), sure tasks (>= CONFIDENT) last; any
    error falls back to file order. Commit: smoke tasks (same rule) then keys30 in list order."""
    keys = list(TEST)
    conf = {}
    try:
        sys.path.insert(0, str(HERE))
        import arc_v6_config as cfg
        conf = cfg.task_confidence(keys, A.outputs, TEST, time.time() + 300)
    except Exception as e:  # noqa: BLE001
        S["order_error"] = f"{type(e).__name__}: {e}"[:300]
    rank = {k: i for i, k in enumerate(keys)}
    if A.commit:
        smoke = sorted((k for k in keys if k in COVERED), key=lambda k: (conf.get(k, float("-inf")), rank[k]))
        order = smoke + [k for k in KEYS30 if k in TEST and k not in smoke]
    else:
        unsure = [k for k in keys if not conf.get(k, float("-inf")) >= CONFIDENT]
        sure = [k for k in keys if conf.get(k, float("-inf")) >= CONFIDENT]
        order = sorted(unsure, key=lambda k: (conf.get(k, float("-inf")), rank[k])) + sure
    S["order"] = {"n": len(order), "first": order[:10], "conf_known": sum(k in conf for k in order),
                  "sure": sum(conf.get(k, float("-inf")) >= CONFIDENT for k in order)}
    return order


def stream_socket(r):
    """The socket under a streaming requests.Response (http.client path; stdlib, stable across urllib3 versions)."""
    for path in (("raw", "_fp", "fp", "raw", "_sock"), ("raw", "_connection", "sock")):
        obj = r
        try:
            for name in path:
                obj = getattr(obj, name)
            if obj is not None:
                return obj
        except AttributeError:
            continue
    return None


def abort_streams():
    """At the abort time: stop the channel and shut down every open stream's socket, so reads blocked on a stream that
    sends nothing (a preempted request) return now; 90 s later the server itself is killed as a last resort."""
    STOP.set()
    with LOCK:
        active = list(ACTIVE.values())
    for r in active:
        try:
            stream_socket(r).shutdown(socket.SHUT_RDWR)
        except Exception:  # noqa: BLE001
            pass
        try:
            r.close()
        except Exception:  # noqa: BLE001
            pass
    with LOCK:
        S["aborted_streams"] = len(active)
    last = threading.Timer(90, lambda: kill(SERVER[0]))
    last.daemon = True
    last.start()


def load_jobs(path):
    """E3 (--jobs): a JSON list of [task, rep] pairs, each run exactly once in this order; any malformed entry, unknown
    task, rep outside [0, MAX_REPS) or duplicate skips the phase before the install (no GPU time spent)."""
    try:
        jobs = [(k, r) for k, r in json.loads(Path(path).read_text())]
    except Exception as e:  # noqa: BLE001
        raise Skip(f"jobs file unreadable: {type(e).__name__}: {e}"[:300])
    bad = [j for j in jobs if j[0] not in TEST or type(j[1]) is not int or not 0 <= j[1] < MAX_REPS]
    if not jobs or bad or len(set(jobs)) != len(jobs):
        raise Skip(f"jobs file invalid: n={len(jobs)} bad={bad[:3]} duplicates={len(jobs) - len(set(jobs))}")
    return jobs


def phase_jobs(jobs, requeue=True):
    """Workers pull (task, rep) from a queue that starts as `jobs`. Production (rep 0 of every task, requeue): a job
    that ends without a verified program queues the same task's next rep at the back, so a task is never in flight
    twice and later reps only go to tasks still unsolved. E3 (--jobs, no requeue): every listed job runs once, whatever
    the other reps of its task found (independent seeds; two reps of one task may be in flight together)."""
    from collections import deque
    from concurrent.futures import ThreadPoolExecutor
    queue = deque(jobs)
    cond = threading.Condition()
    inflight = [0]
    done = []

    def worker():
        while True:
            with cond:
                while not queue and inflight[0] and not STOP.is_set() and time.time() < A.stop_new_at:
                    cond.wait(5)
                if not queue or STOP.is_set() or time.time() >= A.stop_new_at:
                    return
                k, rep = queue.popleft()
                inflight[0] += 1
            row = {"verified": False}
            try:
                row = run_job(k, rep)
                done.append(row)
            except Exception as e:  # noqa: BLE001  (one bad job never stops the channel)
                append("v13-programs.jsonl", {"task": k, "rep": rep, "state": "job_exception",
                                              "error": repr(e)[-300:], "verified": False})
            with cond:
                inflight[0] -= 1
                if requeue and not row.get("verified") and rep + 1 < MAX_REPS and row.get("state") != "prompt_too_long":
                    queue.append((k, rep + 1))
                cond.notify_all()

    with ThreadPoolExecutor(max_workers=A.conc) as pool:
        for _ in range(A.conc):
            pool.submit(worker)
    return done


def poller():
    keep = ("vllm:generation_tokens_total", "vllm:prompt_tokens_total", "vllm:num_requests_running",
            "vllm:num_requests_waiting", "vllm:kv_cache_usage_perc", "vllm:num_preemptions_total",
            "vllm:spec_decode_num_accepted_tokens_total", "vllm:spec_decode_num_draft_tokens_total",
            "vllm:prefix_cache_hits_total", "vllm:prefix_cache_queries_total")
    while not STOP.wait(30):
        try:
            code, text = http("GET", "/metrics", timeout=10)
            vals = {}
            for line in text.splitlines():
                m = re.match(r"([a-zA-Z_:]+)(\{[^}]*\})?\s+([0-9.eE+-]+)$", line)
                if m and m.group(1) in keep:
                    vals[m.group(1)] = vals.get(m.group(1), 0.0) + float(m.group(3))
            append("v13-metrics.jsonl", {"t": el(), **vals})
        except Exception:  # noqa: BLE001
            pass


def summarize(rows):
    done = [r for r in rows if r.get("state") not in (None, "prompt_too_long")]
    ver = [r for r in rows if r.get("verified")]
    s = {"jobs": len(rows), "finished_stop": sum(r.get("finish_reason") == "stop" for r in done),
         "finished_length": sum(r.get("finish_reason") == "length" for r in done),
         "aborted": sum(r.get("state") == "aborted" for r in done),
         "verified": len(ver), "verified_from_reasoning": sum(r.get("source") == "reasoning" for r in ver),
         "forced": sum(bool(r.get("forced")) for r in rows),
         "forced_ok": sum(bool(r.get("forced")) and not r.get("force_error") for r in rows),
         "verified_from_forced": sum(r.get("source") == "forced" for r in ver),
         "verified_tasks": len({r["task"] for r in ver}),
         "tokens": sum((r.get("completion_tokens") or 0) + (r.get("cont_completion_tokens") or 0) for r in rows)}
    if SOLUTIONS:
        s["verified_correct"] = sum(all(r.get("correct") or [False]) for r in ver)
        s["verified_correct_from_reasoning"] = sum(all(r.get("correct") or [False]) for r in ver
                                                   if r.get("source") == "reasoning")
        s["verified_correct_from_forced"] = sum(all(r.get("correct") or [False]) for r in ver
                                                if r.get("source") == "forced")
    return s


TEST, BASE, TASKS, SOLUTIONS, KEYS30, COVERED, OUT_NAME = {}, {}, {}, {}, [], set(), "submission.json"


def load_tasks():
    global TEST, BASE, TASKS, SOLUTIONS, KEYS30, COVERED, OUT_NAME
    comp = Path(A.comp)
    if A.commit:  # dev measurement on public eval: smoke tasks + keys30, scored against the eval solutions
        TEST = json.loads((comp / "arc-agi_evaluation_challenges.json").read_text())
        sol = comp / "arc-agi_evaluation_solutions.json"
        SOLUTIONS = json.loads(sol.read_text()) if sol.exists() else {}
        KEYS30 = json.loads((HERE / "keys30.json").read_text())
        base_path = Path(A.work) / "eval-smoke-submission.json"
        OUT_NAME = "eval-v13-submission.json"
    else:
        TEST = json.loads((comp / "arc-agi_test_challenges.json").read_text())
        base_path = Path(A.work) / "submission.json"
        OUT_NAME = "submission.json"
    try:
        BASE = json.loads(base_path.read_text())
        if not A.commit:
            M.validate(TEST, BASE)
    except Exception as e:  # noqa: BLE001
        if not A.commit:  # never merge onto an unreadable NVARC submission (placeholders would replace its answers)
            raise Skip(f"NVARC base unreadable/invalid: {type(e).__name__}: {e}"[:300])
        BASE = {}
    TASKS = TEST
    try:
        COVERED = {n.split(".")[0].rsplit("_", 1)[0] for n in os.listdir(A.outputs) if not n.startswith("._tmp_")}
    except OSError:
        COVERED = set()
    if not A.commit:  # the merge base for finalize(); written once, before any program answer
        M.atomic_json(os.path.join(A.work, "v13-base-submission.json"), BASE)


def main():
    global A
    ap = argparse.ArgumentParser()
    ap.add_argument("--end-time", type=float, required=True)
    ap.add_argument("--work", default="/kaggle/working")
    ap.add_argument("--outputs", default=os.environ.get("ARC_V6_OUTPUTS", "/kaggle/inference_outputs"))
    ap.add_argument("--input-root", default=os.environ.get("ARC_V6_INPUT_ROOT", "/kaggle/input"))
    ap.add_argument("--tmp", default="/tmp/v13")
    ap.add_argument("--comp-dir", default=None)  # the notebook passes cfg.competition_dir()
    ap.add_argument("--conc", type=int, default=32)
    ap.add_argument("--cap", type=int, default=None)
    ap.add_argument("--effort", default="xhigh")
    ap.add_argument("--force-at", type=int, default=0)  # v3 budget forcing: first request's max_tokens (0 = off)
    ap.add_argument("--answer-tokens", type=int, default=8000)  # v3: max tokens of the forced continuation
    ap.add_argument("--jobs", default=None)  # E3: JSON [[task, rep], ...] run once each in order (no requeue)
    ap.add_argument("--fake-server", default=None)  # test seams: stub server and shorter margins
    ap.add_argument("--need-seconds", type=float, default=None)
    ap.add_argument("--abort-margin", type=float, default=6 * 60)
    ap.add_argument("--stop-margin", type=float, default=None)
    A = ap.parse_args()
    A.commit = not bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
    A.cap = A.cap or 64000  # v2: commit = rerun cap (v1 dev run at 20k: 0/66 natural stops; P0b round-1 p50 = 64k)
    if A.force_at and not 1000 <= A.force_at <= A.cap - 1000:  # else every forcing hits "no room": a silent no-op
        ap.error(f"--force-at {A.force_at} must be in [1000, cap - 1000 = {A.cap - 1000}]")
    A.abort_at = A.end_time - A.abort_margin  # in-flight streams are cut here; their reasoning drafts are still tried
    A.stop_new_at = A.abort_at - (A.stop_margin if A.stop_margin is not None else (5 if A.commit else 15) * 60)
    S.update(commit=A.commit, cap=A.cap, conc=A.conc, effort=A.effort, force_at=A.force_at, answer_tokens=A.answer_tokens,
             end_time=A.end_time,
             seconds_left=round(A.end_time - time.time()))
    proc = None
    stop_poll = None
    try:
        A.comp = A.comp_dir
        if not A.comp:
            comps = [c for c in find("arc-agi_test_challenges.json") if "arc-prize" in c] or \
                find("arc-agi_test_challenges.json")
            A.comp = os.path.dirname(comps[0]) if comps else None
        if not A.comp or not os.path.exists(os.path.join(A.comp, "arc-agi_test_challenges.json")):
            raise Skip("competition files missing")
        load_tasks()
        jobs = load_jobs(A.jobs) if A.jobs else None
        need = A.need_seconds if A.need_seconds is not None else (25 if A.commit else 45) * 60
        A.setup_deadline = A.stop_new_at - need / 3  # server ready by then, else skip (>= need/3 of job time left)
        if A.stop_new_at - time.time() < need:
            raise Skip(f"not enough time: {round(A.stop_new_at - time.time())}s before the stop")
        wheels, tgz, model = phase_inputs()
        phase_gpus()
        if A.commit and not A.fake_server:
            S["phases"]["model_hash"] = {}
            threading.Thread(target=model_hash, args=(model, S["phases"]["model_hash"]), daemon=True).start()
        vpy = sys.executable if A.fake_server else phase_install(wheels, tgz)
        proc = phase_server(vpy, model, A.setup_deadline)
        SERVER[0] = proc
        kv = (S["phases"]["server"].get("chosen") or {}).get("kv_tokens")
        if kv:  # KV-bound concurrency (v1 dev run: KV peaked at 0.64 with 22 streams at cap 20k -> 0.35*cap average)
            A.conc = max(8, min(A.conc, int(kv / (0.35 * A.cap + 6000)) + 2))
        S["conc_effective"] = A.conc
        if A.force_at:  # visible at server start, not after the generation window
            S["token_id_parity"] = token_id_parity()
            log(f"token id parity {json.dumps(S['token_id_parity'])}")
            if not S["token_id_parity"]["ok"]:
                S["force_disabled"] = "token id parity failed"
                A.force_at = 0
        stop_poll = threading.Thread(target=poller, daemon=True)
        stop_poll.start()
        S["sandbox_selftest"] = P.run_program("import numpy as np\ndef transform(g):\n    return np.array(g).T.tolist()\n",
                                              [[[1, 2]]])
        if jobs:
            S["order"] = {"n": len(jobs), "first": jobs[:10], "jobs_file": A.jobs, "requeue": False}
        else:
            jobs = [(k, 0) for k in job_order()]
        S["gen_start"] = el()
        save()
        log(f"jobs start: {len(jobs)} jobs, conc {A.conc}, cap {A.cap}, stop_new in "
            f"{round(A.stop_new_at - time.time())}s")
        watchdog = threading.Timer(max(0, A.abort_at - time.time()), abort_streams)
        watchdog.daemon = True
        watchdog.start()
        rows = phase_jobs(jobs, requeue=not A.jobs)
        S["gen_seconds"] = round(el() - S["gen_start"])
        S["summary"] = summarize(rows)
        log("summary " + json.dumps(S["summary"]))
    except Skip as e:
        S["skipped"] = str(e)
        log(f"skipped: {e}")
    except Exception as e:  # noqa: BLE001
        import traceback
        S["skipped"] = f"error {type(e).__name__}: {e}"[:500]
        S["traceback"] = traceback.format_exc()[-3000:]
        log(f"error: {e}")
    finally:
        STOP.set()
        kill(proc)
        if RESULTS:
            try:
                write_submission()
            except Exception as e:  # noqa: BLE001
                S["final_write_error"] = repr(e)[:300]
        for log_path in glob.glob(os.path.join(A.work, "v13-server-*.log")):  # keep outputs small (P1: 872 MB log)
            try:
                if os.path.getsize(log_path) > 50 << 20:
                    with open(log_path, "rb") as f:
                        f.seek(-(20 << 20), 2)
                        tail = f.read()
                    with open(log_path, "wb") as f:
                        f.write(b"[truncated to last 20 MB]\n" + tail)
            except Exception:  # noqa: BLE001
                pass
        S["elapsed"] = el()
        save()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile keys30.json
["dbff022c", "3dc255db", "2b83f449", "5545f144", "dd6b8c4b", "f931b4a8", "53fb4810", "332f06d7", "d35bdbdc", "58f5dbd5", "faa9f03d", "dfadab01", "80a900e0", "581f7754", "446ef5d2", "7b0280bc", "a251c730", "2c181942", "4e34c42c", "7ed72f31", "b6f77b65", "38007db0", "4a21e3da", "142ca369", "cb2d8a2c", "6e4f6532", "13e47133", "e12f9a14", "3e6067c3", "e3721c99"]

In [ ]:
%%writefile qwen38_fp8_sha256.json
{"source":"https://huggingface.co/api/models/Qwen/Qwen3.8-27B-FP8/tree/017b9c7af6b5689d5dd426a76e0bc077eb5ca20a","fetched_at":"2026-10-12T08:05Z","revision":"017b9c7af6b5689d5dd426a76e0bc077eb5ca20a","kaggle_mirror":"mikedan7/qwen3-8-27b-fp8-official/PyTorch/hf-fp8/1 (same file names and sizes; README says unmodified mirror at revision 017b9c7af6b5)","files":{"layers-0.safetensors":[383865448,"07f700e293baeaf3cd4240c3df1a948c4403f16961ea7979e86c8d6a9f8fd466"],"layers-1.safetensors":[383865448,"35840b5d452c6e438d000e5c1d8d1bc793d257394404689b8f9424749eee8edc"],"layers-10.safetensors":[383865472,"ab5fbc076ccd6514aba3ab67a0ac1d6701ad395d40cccc0086cf0284ef5d68c9"],"layers-11.safetensors":[372313760,"076b4ba44e9311c8fb8fdc0045f8ce9fd0c75584ade4e95510056b7db4b7ec61"],"layers-12.safetensors":[383865472,"86d8ec0b8fc8e3ffbdfd667b97050533ddf5a4d273b4fe1190a4335542c52829"],"layers-13.safetensors":[383865472,"71dd986134f4338a7b8290ed24993ad34a0f77b76a4cc060fe6403e36dcdda5b"],"layers-14.safetensors":[383865472,"95b285915cba994917b527030874227382ea3421dc7f47d804c0ca5851f039fb"],"layers-15.safetensors":[372313760,"1e87d61e77d2b802f796ab8a049b1119648c9b70d2ba1067f456e5a04449cfc3"],"layers-16.safetensors":[383865472,"8bcdbf4e2c7a8dda3043c4453a476b30a582d58aabfaa49b2829a8b66a664acb"],"layers-17.safetensors":[383865472,"571cea44717878cc8cbbfbd544acfe2d155f5b963fbf5dc67b3fa96afe840a1b"],"layers-18.safetensors":[383865472,"d8700627ca2267fb7beb5d398d6a659a9bfeeb2af76f10e6827b1e4359d59286"],"layers-19.safetensors":[372313760,"2b24cfb752bbed959c614d7fd85412ee27ab2ac1f47deef5d09d0592781400e4"],"layers-2.safetensors":[383865448,"32e63c6455ebfa10b4c36af2878eefcb6f8ccf8e947a62575444dc412c38ac1a"],"layers-20.safetensors":[383865472,"52fd00c68d4f1df96ba701290c04237257ebb832879bb0b6a629525809d84a64"],"layers-21.safetensors":[383865472,"e3c29dd949808c13116d853ce4b0f96cb1238fc0cb6999ab76735deef6bda77a"],"layers-22.safetensors":[383865472,"63446546e3e09c2394fc51855f4e390cb47f896bb8b744f3a210c3159a3ccaf2"],"layers-23.safetensors":[372313760,"4166ed11d0329985f9813ba6a70474ea9f5aeff3d97db3d7b2e81185bd41a996"],"layers-24.safetensors":[383865472,"720bdcfe12b6627bed29feda9c96256e15c20602704669d05097668a9f03dd45"],"layers-25.safetensors":[383865472,"1b188e7e4e8ae6f5d753c597f61d86c1593495de4307c273d8dda9946ec48bb4"],"layers-26.safetensors":[383865472,"ad8c3e3d4b79dfedad674318ee4b150b2cd78b56747c7c8d88a65818109a545d"],"layers-27.safetensors":[372313760,"8d63f461430045960f27a22abf685bfc7445fffbb24764e4a7fa993ba7edac08"],"layers-28.safetensors":[383865472,"f78557cda66107ce25f8c5ecde2f83da874b3bf7c50381e062e14e7294a5b64d"],"layers-29.safetensors":[383865472,"09f264d6fdddab0fb5ecce18034a9a01944d15118147c97058fba101a60f8af1"],"layers-3.safetensors":[372313744,"302f9af90bb683a8be9e96d124b470a2eddee6612c95c39a8e93f26eb654563d"],"layers-30.safetensors":[383865472,"54afe4fc7262ee597d4d9b80d2d6fe8ae501846ff124334b575bed4c6850ed97"],"layers-31.safetensors":[372313760,"2779338203d0715ca323ac0b7fdbb0d450ce2bd1694be395a211a02ade3f34d9"],"layers-32.safetensors":[383865472,"f4af8893594458b6cdb074ef20f9accb708361fe904012465990d09e74994509"],"layers-33.safetensors":[383865472,"8f038505b9fc7d0b31ca976c728b56028ec28390a15ca0410f00df18d3abc1a4"],"layers-34.safetensors":[383865472,"0b28f43cc4d3ae50e37eb3c2ce12a6e181b1c7586a27d5a05c6a3367f08286c9"],"layers-35.safetensors":[372313760,"535b61b9eabbcda5de20e21d2d46fb52528cc9c1f8fac38920382ea2f1030837"],"layers-36.safetensors":[383865472,"2babd209cafbe12a6e375ff6e7a1029f36ee8226f2a9babf6646630f31dd3fb8"],"layers-37.safetensors":[383865472,"651cae579808983fefec954db6e2ac8c3f98bfb038ff1d50aef0505ba7394e65"],"layers-38.safetensors":[383865472,"1c60932dec650cc7679121eaa4d90b8d10a600bf58e23b0784588a339469dd0a"],"layers-39.safetensors":[372313760,"b0c7df4d51b99637ff4ea35eebf7eb134db592eb943e3ccde921729f33949058"],"layers-4.safetensors":[383865448,"b7f367125cdb4b3c3920d1d3ed1ba0d73e464c3e3a92dbd1c195658cb9200afd"],"layers-40.safetensors":[383865472,"6908f1ab1a3d7828a566099ecf393582bec988d0f82f6141848942d1494b0321"],"layers-41.safetensors":[383865472,"21ee5d9842074888bd9fdffa875d1f29f8c77f1f5ccbb8f53becb188e1749caa"],"layers-42.safetensors":[383865472,"565379d4291c06cdf9a4f66d5d21385848a1e5ab17b55fea6c1c8e544f8498f0"],"layers-43.safetensors":[372313760,"4bb8c0a4fadda1f1d66ef6f0dde5a5a94001d04af1842e42009833aff70bf416"],"layers-44.safetensors":[383865472,"14339304cd520a8673f179cb8352c36fbebe184f15309aee2a79afd2d4476fef"],"layers-45.safetensors":[383865472,"7a076aca542f937cc87dec92ab49fd782eae234642b40bc706a6108b3dff2c2a"],"layers-46.safetensors":[383865472,"d82d472cfc4792012934467b90c5e0242df4867a430dd651d13cdee72b7c6645"],"layers-47.safetensors":[372313760,"a83fc757fb11bc50ed3a0c9998b2b2f55c3b6a07598ed458bdbece4285ffaf1f"],"layers-48.safetensors":[383865472,"f6aa126d006c7d976c20b1330c118b42cf3bbe8a7d216bb14f2240d5c49bc72f"],"layers-49.safetensors":[383865472,"7ed54c375407c2baec8cfabe8336f8d3782246848298bb232ac2582113676c69"],"layers-5.safetensors":[383865448,"ebbc2c4bd2b98877caafa7073b6ed732a0284eb82269cf6d21710e245af5837a"],"layers-50.safetensors":[383865472,"73825e1f056c3de53276d3c65743e62b67bf3ca082cf7077ecba1ffef18e9bf1"],"layers-51.safetensors":[372313760,"b6abd8b62b5c4b7beca8cdccd306065be2192ef42491c587ca019d971ed86a1f"],"layers-52.safetensors":[383865472,"6251d51e6fbec6f463ca875fcbe852e07c5025145affe9af053a04fbdc37288f"],"layers-53.safetensors":[383865472,"a9f2f43b4978bba2f755c6d2eb39f8561965bc3c1b2fd1073bb363aff11a9b8c"],"layers-54.safetensors":[383865472,"f0246c9974a4b1207f8f5d05a09c54c50ea61fbfd2dc6480d2b76d3647ece292"],"layers-55.safetensors":[372313760,"649c400495616cf888b7c52e538113610a36265c531d84bff484f8149c157405"],"layers-56.safetensors":[383865472,"eb0a8139a36138639a37219c158b9415412b68d316235061a72e57da5509c188"],"layers-57.safetensors":[383865472,"d7616ad33f9256342573ff85b3f2a0e368dd120257c0c7158564587036119d95"],"layers-58.safetensors":[383865472,"a8dff0e37d0e3903cc29711b935c2865ccd38bd930e7487ae2f4bedbd18ff185"],"layers-59.safetensors":[372313760,"573ed65fd455ec9bd811e95946ea422e08ac36860fdff6968a6d108263a54175"],"layers-6.safetensors":[383865448,"2fea2aaa61d566ba8af27ee89d29d9ee9f1c7f6fbe692772d9ba61fc45109ef6"],"layers-60.safetensors":[383865472,"9a19c30b190bf552eb424af16103f610bcd944fb276b3275eb7c73849799b7de"],"layers-61.safetensors":[383865472,"ff744debd4dd0eef450ed24473df57680db404914302e7147a3b72154131f4a0"],"layers-62.safetensors":[383865472,"d9b3b9e472f78bcadf62c446084833bb549faff9e730b21219f1f80d583eff23"],"layers-63.safetensors":[372313760,"59ba4a3af5e6bc2008c2d6b5d9be8eeaf255cddc0aaf8a0d9583b496bb2e3ae0"],"layers-7.safetensors":[372313744,"f2e0137e878016a7afdb2314972e0cbbe2461a67b00eb1d691b4de8ea2665da8"],"layers-8.safetensors":[383865448,"1721b7bcd730891c224b70a7b6147e44a574a2d6c4d924760acf97728249c92c"],"layers-9.safetensors":[383865448,"0a21e07065b5bb04adc339300dea1872cc48ab8eab3b02df092f7cbc67c8d6f6"],"mtp.safetensors":[477202224,"e5e4464a3793cc261de536592830bca40e7f3af159ed038c358f5660917cf43b"],"outside.safetensors":[6007102112,"ddff1d6665a2b39f2612fce0ef955e2436724c565bfbcbc127c7ffd078b698ff"],"tokenizer.json":[12809320,"0997f410c57a1f4e53b09e4be8f4a172d90edd9564368fb0847030937229b9f3"]}}


In [ ]:
%%writefile e4-jobs.json
[["e3721c99", 3], ["3e6067c3", 3], ["332f06d7", 3], ["faa9f03d", 3], ["f931b4a8", 3], ["6e4f6532", 3], ["7b0280bc", 3], ["13e47133", 3], ["142ca369", 3], ["e12f9a14", 3]]


In [ ]:
import json, os
assert not os.getenv('KAGGLE_IS_COMPETITION_RERUN'), 'E4 is an offline measurement kernel, never a submission'
e4_end_time = global_start + 7500
print(json.dumps({'e4_seconds_left': round(e4_end_time - time.time())}))

In [ ]:
!PYTHONUNBUFFERED=1 python v13_llm.py --end-time {e4_end_time} --jobs e4-jobs.json --cap 96000

In [ ]:
with open('/kaggle/working/v13-llm.json') as f:
    _s = json.load(f)
print(json.dumps({k: _s.get(k) for k in ('skipped', 'conc_effective', 'cap', 'gen_seconds', 'summary', 'order')}))